### Block 0 – Install dependencies

This cell installs all Python packages needed for the tutorial:
`openai` for model calls, `datasets` for loading the FPB dataset,
`scikit-learn` for evaluation metrics, `tqdm` for progress bars,
and `pandas` for basic table-like data handling.
Run this cell once when you start the notebook.

In [9]:
!pip install -q openai datasets scikit-learn tqdm pandas

### Block 1 – Imports and OpenAI client setup

This block imports all required Python libraries and securely
reads your OpenAI API key using `getpass`.  
It then creates a single `OpenAI` client instance, which will be
reused for all model calls in later blocks.


In [10]:
import os
import getpass

from openai import OpenAI, BadRequestError
from datasets import load_dataset
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd

os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key here: ")

client = OpenAI()


Paste your OpenAI API key here: ··········


### Block 2 – Load the FPB test split and inspect the data

This block downloads the `ChanceFocus/en-fpb` dataset from HuggingFace
and loads the **test** split, which contains 970 labeled sentences.  
It prints basic dataset information and the first example so we can see
the available fields (such as `text` and `answer`) and verify that the
data has been loaded correctly.


In [11]:
dataset = load_dataset("ChanceFocus/en-fpb", split="test")

print(dataset)
print("Number of examples:", len(dataset))
print("Example 0:", dataset[0])


Dataset({
    features: ['id', 'query', 'answer', 'text', 'choices', 'gold'],
    num_rows: 970
})
Number of examples: 970
Example 0: {'id': 'fpb3876', 'query': 'Analyze the sentiment of this statement extracted from a financial news article. Provide your answer as either negative, positive, or neutral.\nText: The new agreement , which expands a long-established cooperation between the companies , involves the transfer of certain engineering and documentation functions from Larox to Etteplan .\nAnswer:', 'answer': 'positive', 'text': 'The new agreement , which expands a long-established cooperation between the companies , involves the transfer of certain engineering and documentation functions from Larox to Etteplan .', 'choices': ['positive', 'neutral', 'negative'], 'gold': 0}


### Block 3 – Define label set and output normalization

This block defines the three sentiment labels used in FPB:
`negative`, `neutral`, and `positive`.  
It also implements a helper function `normalize_prediction`, which
takes the raw text output from the model and maps it to exactly one
of these three labels. This ensures that small variations in the
model's wording still produce a valid class label.


In [12]:
LABELS = ["negative", "neutral", "positive"]

def normalize_prediction(raw: str) -> str:
    text = (raw or "").strip().lower()
    for label in LABELS:
        if label in text:
            return label
    return "neutral"


### Block 4 – o3 sentiment classifier (low reasoning effort + retry)

This block defines how we call the `o3` reasoning model to classify
a single financial news sentence.  
We use the Responses API with a short instruction prompt and set
`reasoning={"effort": "low"}` so that the model uses less internal
reasoning for each request.  
The function `classify_with_o3_low` also includes simple retry logic:
it will automatically retry a few times when transient server errors
(500) occur, and then return a normalized sentiment label.


In [15]:
import time
from openai import BadRequestError, InternalServerError

INSTRUCTIONS = """
You are a financial sentiment classifier.

Given a single sentence from a financial news article, classify its overall
sentiment from the perspective of an investor as exactly one of:
positive, neutral, or negative.

Respond with only one word: positive, neutral, or negative.
"""

def classify_with_o3_low(sentence: str, max_retries: int = 3, backoff: float = 2.0) -> str:
    """
    Call o3 with low reasoning effort.
    Retries on server-side errors (500) up to max_retries times.
    """
    for attempt in range(max_retries):
        try:
            response = client.responses.create(
                model="o3",
                instructions=INSTRUCTIONS,
                input=sentence,
                reasoning={"effort": "low"},
                max_output_tokens=64,
            )
            text = response.output_text
            return normalize_prediction(text)

        except InternalServerError as e:
            # 500 server error: safe to retry
            print(f"[attempt {attempt+1}/{max_retries}] InternalServerError, retrying...")
            print(e)
            if attempt < max_retries - 1:
                time.sleep(backoff * (attempt + 1))
            else:
                # after last attempt, re-raise so caller can decide what to do
                raise

        except BadRequestError as e:
            # 4xx like invalid_request (e.g., max_tokens) we do NOT retry here
            print("BadRequestError:", e)
            raise


### Block 5 – Run evaluation loop over the FPB test set

This block iterates over all 970 examples in the FPB test split.  
For each example, it extracts the input sentence and gold label,
calls `classify_with_o3_low` to get the model prediction, and stores
both the true labels (`y_true`) and predictions (`y_pred`).  
If an error occurs for a particular example (after retries), the index
is recorded in `failed_idx` and that example is skipped so that the
evaluation can continue.


In [16]:
y_true = []
y_pred = []
failed_idx = []

for idx, example in enumerate(tqdm(dataset, total=len(dataset))):
    sentence = example["text"]
    true_label = example["answer"].lower().strip()

    try:
        pred_label = classify_with_o3_low(sentence)
    except BadRequestError as e:
        print(f"Error at example {idx}: {e}")
        failed_idx.append(idx)
        continue

    y_true.append(true_label)
    y_pred.append(pred_label)

print("Total examples:", len(dataset))
print("Successful predictions:", len(y_true))
print("Failed examples:", len(failed_idx))
print("Failed indices:", failed_idx[:20])


 65%|██████▍   | 628/970 [22:59<14:11,  2.49s/it]

[attempt 1/3] InternalServerError, retrying...
Error code: 500 - {'error': {'message': 'An error occurred while processing your request. You can retry your request, or contact us through our help center at help.openai.com if the error persists. Please include the request ID req_9457f3dedc114d53a48d5e768c54c86a in your message.', 'type': 'server_error', 'param': None, 'code': 'server_error'}}


 67%|██████▋   | 648/970 [23:48<09:31,  1.77s/it]

[attempt 1/3] InternalServerError, retrying...
Error code: 500 - {'error': {'message': 'An error occurred while processing your request. You can retry your request, or contact us through our help center at help.openai.com if the error persists. Please include the request ID req_82512877b7aa44cdbde4112162784a52 in your message.', 'type': 'server_error', 'param': None, 'code': 'server_error'}}


 99%|█████████▉| 961/970 [35:48<00:25,  2.87s/it]

[attempt 1/3] InternalServerError, retrying...
Error code: 500 - {'error': {'message': 'An error occurred while processing your request. You can retry your request, or contact us through our help center at help.openai.com if the error persists. Please include the request ID req_8a598d817a224f54a135237e91867857 in your message.', 'type': 'server_error', 'param': None, 'code': 'server_error'}}


100%|██████████| 970/970 [36:14<00:00,  2.24s/it]

Total examples: 970
Successful predictions: 970
Failed examples: 0
Failed indices: []


### Block 6 – Compute evaluation metrics and inspect predictions

This block computes standard classification metrics for the model on
the FPB test set using `scikit-learn`.  
It prints the overall accuracy and a detailed classification report
(precision, recall, and F1-score for each sentiment class).  
Then it builds a `pandas` DataFrame containing the input text, the
gold label, and the model prediction, prints the first 10 predictions,
and finally shows several misclassified examples to help us understand
where the model tends to make mistakes.


In [18]:
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd

# Overall metrics
accuracy = accuracy_score(y_true, y_pred)
print(f"Overall accuracy: {accuracy:.4f}\n")

print("Classification report:")
print(classification_report(y_true, y_pred, labels=LABELS))

# Build a DataFrame with text / gold / pred
df = pd.DataFrame({
    "text": [ex["text"] for ex in dataset],
    "gold": y_true,
    "pred": y_pred,
})

print("\nFirst 10 predictions:")
print(df.head(10).to_string(index=False))

# Error analysis
errors = df[df["gold"] != df["pred"]]
print(f"\nNumber of errors: {len(errors)}")
print("Some error examples:")
print(errors.head(20).to_string(index=False))


Overall accuracy: 0.8165

Classification report:
              precision    recall  f1-score   support

    negative       0.84      0.88      0.86       116
     neutral       0.86      0.82      0.84       577
    positive       0.72      0.78      0.75       277

    accuracy                           0.82       970
   macro avg       0.81      0.83      0.82       970
weighted avg       0.82      0.82      0.82       970


First 10 predictions:
                                                                                                                                                                                                                                                                                 text     gold     pred
                                                                                           The new agreement , which expands a long-established cooperation between the companies , involves the transfer of certain engineering and documentation function